# Data Loading & Cleaning

In [ ]:
# libraries
import pandas as pd
import IPython.display as display
import numpy as np
import os

# utils
from utils.data_io_clean import (
    load_and_clean_broker,
    load_and_clean_nse_eq_master,
    load_and_clean_nse_etf_master,
    load_and_clean_sgb,
)

from utils.dataset_builder import (
    build_canonical_portfolio,
    build_historical_price_dataset,
    build_index_price_dataset,
)

from utils.api_io import (
    fetch_historical_prices_n_years,
)

from utils.features import (
    compute_daily_ret,
    compute_annualized_mean_ret,
    compute_individual_annualized_volatility,
    winsorize_returns,
    compute_covariance_matrix,
    compute_correlation_matrix,
)

from utils.universe import (
    remove_securities,
    remove_securities_leq_weight_w,
)

from utils.preprocessing import (
    fill_with_proxy,
)

from utils.portfolio import (
    portfolio_variance,
    portfolio_volatility,
    portfolio_return,
)

# config
from src import config

np.random.seed(420)

## Loading Initial Data

In [ ]:
nse_eq_master_df = load_and_clean_nse_eq_master(path=config.NSE_EQ_PATH)
nse_etf_master_df = load_and_clean_nse_etf_master(path=config.NSE_ETF_PATH)
current_portfolio = load_and_clean_broker(path=config.PORTFOLIO_PATH)
sgb_price_df = load_and_clean_sgb(path=config.SGB_PATH)

display.display(nse_eq_master_df.tail(5))
display.display(nse_etf_master_df.tail(5))
display.display(sgb_price_df.tail(5))
display.display(current_portfolio)

## Build Primary Datasets

### Convert Broker Data into Canonical Format

In [ ]:
canon = build_canonical_portfolio(
    portfolio_df=current_portfolio,
    nse_eq_df=nse_eq_master_df,
    nse_etf_df=nse_etf_master_df,
    sgb_df=sgb_price_df,
)

canon

We now have a canonical portfolio dataframe that merges broker holdings with NSE master security metadata and aligns each holding with its price history. This unified view will be the basis for return, risk, and portfolio analytics.

Some holdings may have limited historical coverage or be outside the scope of this analysis (e.g., small actively managed mutual funds), so they may be treated as exceptions or excluded when they cannot be reasonably modeled.

### Build Historical Price Dataset (10Y)

In [ ]:
equities_3y_price_df = fetch_historical_prices_n_years(
    symbols=canon["symbol"].tolist(),
    n_years=10,
)

price_history_df = build_historical_price_dataset(
    eq_etf_historical=equities_3y_price_df,
    sgb_df=sgb_price_df,
)

price_history_df

### Analyze Historical Price Dataset

In [ ]:
price_history_df.describe()

In [ ]:
price_history_df.isna().sum()

In [ ]:
for col in price_history_df.columns:
    print(f"{str(col)}: {price_history_df[col].first_valid_index()}")

In [ ]:
print(canon[["symbol", "weight"]])

### Notes on limited price histories
Some securities in our universe have short or patchy historical price coverage. Dropping these securities would significantly reduce the available data, so we prefer to use proxy return series where appropriate. If a security cannot be reasonably proxied, we may choose to exclude it from the analysis.

In [ ]:
# mosmall_weight = canon[canon["symbol"] == "MOSMALL250.NS"]["weight"].iloc[0]
# canon = canon[canon["symbol"] != "MOSMALL250.NS"].copy()
# canon["weight"] /= 1 - mosmall_weight
# canon

In [ ]:
# canon["weight"].sum()

In [ ]:
# price_history_df = price_history_df.drop(columns="MOSMALL250.NS")
# price_history_df

In [ ]:
# price_history_df.info

### Mitigating missing data with proxy returns
Some securities (e.g., newer ETFs) have very short price histories. Dropping rows with missing values would discard a large portion of our dataset. To preserve data, we will use proxy return series (typically index returns) for securities with insufficient history.

In [ ]:
index_price_df = build_index_price_dataset()
index_price_df

### Fill missing returns using proxy mappings
We map symbols with short histories to appropriate proxy indexes so that we maintain a consistent return matrix for downstream analytics.

Some symbols (e.g., MONIFTY500) have short histories but represent broad indexes. Where it makes sense, we proxy these securities with the returns of the underlying index (e.g., TRI index). Any symbols without a viable proxy may need to be excluded to keep the dataset consistent.

### Preprocessing workflow
With the return series prepared (including proxies), we perform the following steps:
- Calculate daily returns.
- Detect and mitigate extreme outliers.
- Compute individual annualized volatility.
- Compute pairwise correlations.
- Build a covariance matrix using a composite estimator (correlations + volatilities), applying eigenvalue clipping to ensure the covariance matrix is positive semi-definite.

In [ ]:
ret_df = compute_daily_ret(price_history=price_history_df)

ret_df

### Inspect return distributions for outliers
We examine the return series to identify any extreme values that could distort risk estimates.

In [ ]:
index_ret_df = compute_daily_ret(index_price_df)
index_ret_df

In [ ]:
ret_df = fill_with_proxy(
    ret=ret_df,
    proxy_rets=index_ret_df,
    proxy_map={
        "MONIFTY500.NS": "NIFTY 500",
        "NEXT50.NS": "NIFTY NEXT 50",
        "MOSMALL250.NS": "NIFTY SMALLCAP 250",
        "ITETF.NS": "NIFTY IT",
    },
)
ret_df.head()

In [ ]:
ret_df.describe()

Some securities do seem to have massive outliers judging by max returns over 800%. Let's also look at the annualized std.

In [ ]:
compute_individual_annualized_volatility(ret=ret_df)

### Winsorize extreme return values
We use MAD-based Winsorization with a threshold of 8 to cap extreme return observations while leaving the bulk of the distribution intact.

In [ ]:
ret_df = winsorize_returns(ret=ret_df, k=8)
display.display(ret_df)
display.display(ret_df.describe())

In [ ]:
covariance_matrix = compute_covariance_matrix(ret=ret_df)
covariance_matrix

In [ ]:
portfolio_variance()